In [8]:
from ultralytics import YOLO
# --- CONFIGURATION ---
model = YOLO('yolo11x-pose.pt')        # Le chemin vers votre modèle entraîné
video_path = '/Users/philomenecarrel/4A/projet_info_ping_pong/PingPongTracking/roboflow/datas/video2.mp4'   # Votre vidéo de match
# --------------------- 
results = model(video_path, vid_stride=32, show=True)  # Affiche les résultats en temps réel


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/56) /Users/philomenecarrel/4A/projet_info_ping_pong/PingPongTracking/roboflow/datas/video2.mp4: 480x640 1 person, 433.6ms
video 1/1 (frame 2/56) /Users/philomenecarrel/4A/projet_info_ping_pong/PingPongTracking/roboflow/datas/video2.mp4: 480x640 1 person, 470.1ms
video 1/1 (frame 3/56) /Users/philomenecarrel/4A/projet_info_ping_pong/PingPongTracking/roboflow/datas/video2.mp4: 480x640 1 person, 398.9ms
video 1/1 (frame 4/56) /Users/philo

In [9]:
def classify_stroke(pose_results, dominant_hand="right"):
    # Get keypoints (x, y)
    if pose_results[0].keypoints is not None:
        kp = pose_results[0].keypoints.xy[0]

        # 1. Get Torso Center (average of shoulders)
        shoulder_l, shoulder_r = kp[5], kp[6]
        torso_center_x = (shoulder_l[0] + shoulder_r[0]) / 2

        # 2. Get Hitting Wrist
        wrist_idx = 10 if dominant_hand == "right" else 9
        wrist_x = kp[wrist_idx][0]

        # 2. Get Hitting shoulder

        shoulder_x = shoulder_r[0] if dominant_hand == "right" else shoulder_l[0]

        # 3. Logic based on side
        if dominant_hand == "right":
            if wrist_x > shoulder_x:
                return "Forehand"
            else:
                return "Backhand"
        else: # Left-handed logic (inverted)
            if wrist_x < shoulder_x:
                return "Forehand"
            else:
                return "Backhand"

            

In [7]:
classify_stroke(results, dominant_hand="right")

'Backhand'

In [ ]:
print(results[0].boxes)  

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([0.])
conf: tensor([0.9203])
data: tensor([[3.0482e+02, 1.1975e-01, 4.5957e+02, 2.3660e+02, 9.2029e-01, 0.0000e+00]])
id: None
is_track: False
orig_shape: (480, 640)
shape: torch.Size([1, 6])
xywh: tensor([[382.1946, 118.3584, 154.7584, 236.4774]])
xywhn: tensor([[0.5972, 0.2466, 0.2418, 0.4927]])
xyxy: tensor([[3.0482e+02, 1.1975e-01, 4.5957e+02, 2.3660e+02]])
xyxyn: tensor([[4.7627e-01, 2.4948e-04, 7.1808e-01, 4.9291e-01]])


In [6]:
def boxes_intersect(box_ball, box_racket):
    # box format: [x1, y1, x2, y2]
    # Vérifie si une boîte est à gauche, à droite, au-dessus ou en dessous de l'autre
    return not (box_ball[2] < box_racket[0] or # Balle à gauche de raquette
                box_ball[0] > box_racket[2] or # Balle à droite de raquette
                box_ball[3] < box_racket[1] or # Balle au-dessus de raquette
                box_ball[1] > box_racket[3])   # Balle en dessous de raquette

In [ ]:
import cv2
from ultralytics import YOLO

# 1. Charger les deux modèles
model_obj = YOLO('models/best-2.pt')
model_pose = YOLO('models/yolo11n-pose.pt')

cap = cv2.VideoCapture("datas/video2.mp4")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # 2. Détection d'objets (Balle et Raquette)
    results_obj = model_obj(frame,conf=0.6, iou=0.5, verbose=False)[0]
    
    ball_box = None
    racket_box = None

    # Extraction des boîtes (IDs à vérifier selon ton entraînement)
    for box in results_obj.boxes:
        cls = int(box.cls[0])
        if cls == 0: ball_box = box.xyxy[0].cpu().numpy()   # Classe Balle
        if cls == 1: racket_box = box.xyxy[0].cpu().numpy() # Classe Raquette

    # 3. Vérification de la superposition
    if ball_box is not None and racket_box is not None:
        if boxes_intersect(ball_box, racket_box):
            
            # --- ÉTAPE CRUCIALE : On ne demande la POSE que si ça se touche ---
            results_pose = model_pose(frame, verbose=False)[0]
            
            if results_pose.keypoints:
                # Analyse de la pose pour classer le coup
                stroke = classify_stroke(results_pose)
                print(f"IMPACT DÉTECTÉ : {stroke}")
                
                # Optionnel : Dessiner le résultat sur l'image
                cv2.putText(frame, stroke, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Arbitrage", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [16]:
import cv2
from ultralytics import YOLO

# --- CONFIGURATION ---
# Remplace par tes chemins de modèles
model_obj = YOLO('models/best-2.pt') 
model_pose = YOLO('models/yolo11n-pose.pt')

def boxes_intersect(box_ball, box_racket):
    # box format: [x1, y1, x2, y2]
    # Vérifie si une boîte est à gauche, à droite, au-dessus ou en dessous de l'autre
    return not (box_ball[2] < box_racket[0] or # Balle à gauche de raquette
                box_ball[0] > box_racket[2] or # Balle à droite de raquette
                box_ball[3] < box_racket[1] or # Balle au-dessus de raquette
                box_ball[1] > box_racket[3])   # Balle en dessous de raquette

def classifier_coup(keypoints):
    """ Analyse la position du poignet par rapport au centre du torse """
    # Points YOLO : 5=Épaule G, 6=Épaule D, 10=Poignet D (pour un droitier)
    kp = keypoints.xy[0].cpu().numpy()
    if len(kp) < 11: return "Inconnu"

    centre_torse_x = (kp[5][0] + kp[6][0]) / 2
    poignet_x = kp[10][0] # On suit la main droite

    # Si le poignet est à droite du coude = Coup Droit, sinon = Revers
    return "COUP DROIT" if poignet_x < kp[8][0] else "REVERS"

# --- BOUCLE VIDÉO ---
cap = cv2.VideoCapture("datas/video2.mp4")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # 1. Détection de la Balle et de la Raquette
    results_obj = model_obj(frame, verbose=False, conf=0.3)[0]
    
    ball_box = None
    racket_box = None

    for box in results_obj.boxes:
        cls = int(box.cls[0])
        # Ajuste les IDs (0 et 1) selon les classes de ton modèle entraîné
        if cls == 0: ball_box = box.xyxy[0].cpu().numpy()
        if cls == 1: racket_box = box.xyxy[0].cpu().numpy()

    # 2. Déclenchement conditionnel via TA fonction
    if ball_box is not None and racket_box is not None:
        if boxes_intersect(ball_box, racket_box):
            
            # 3. L'impact est détecté -> On demande la pose
            results_pose = model_pose(frame, verbose=False)[0]
            
            if results_pose.keypoints:
                coup = classifier_coup(results_pose.keypoints)
                
                # Affichage des résultats
                cv2.rectangle(frame, (int(ball_box[0]), int(ball_box[1])), 
                              (int(ball_box[2]), int(ball_box[3])), (0, 255, 0), 2)
                cv2.putText(frame, f"IMPACT : {coup}", (50, 80), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
                
                # Optionnel : Dessiner le squelette uniquement à l'impact
                frame = results_pose.plot(img=frame, boxes=False)

    cv2.imshow("Arbitrage Automatique", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()